# Phase 2: Structural Validation (ESMFold + Foldseek), Catalytic Triad 3D Check, and Biased ICL
This notebook performs:
1. **ESMFold Structural Prediction:** Folds the 71 novel de novo PETase designs from Experiment 1 and filters by $\text{mean pLDDT} \ge 80$.
2. **Foldseek Structural Alignment:** Compares folds against **IsPETase** (PDB: `6EQE`) and **LCC** (PDB: `4EB0`).
3. **3D Catalytic Triad Geometry Verification:** Measures spatial distances between **Ser-His-Asp** to confirm catalytic triad integrity.
4. **Statistical Scaling ($N\_PROMPTS = 25$):** Tighter error bars on Evo 2 ICL yield (capping at $n=64$; omitting $n=128$ to avoid OOM).
5. **Activity-Biased In-Context Conditioning:** Curates elite high-activity & heat-tolerant PANDA exemplars for targeted generation.


## 1. Environment & GPU Verification


In [ ]:
import subprocess, os, sys, pathlib

# Check GPU
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], 
                         capture_output=True, text=True).stdout)
except Exception as e:
    print("Warning: nvidia-smi failed:", e)

# Mount Google Drive if running in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")



## 2. Install Dependencies & Tools (Transformers, Foldseek, Reference PDBs)


In [ ]:
# Install transformers, accelerate, biopython
!pip install -q transformers accelerate biopython matplotlib pandas numpy

# Download and unpack static Foldseek binary (Linux AVX2)
import urllib.request, tarfile, shutil

FOLDSEEK_URL = "https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz"
if not pathlib.Path("foldseek/bin/foldseek").exists():
    print("Downloading Foldseek...")
    urllib.request.urlretrieve(FOLDSEEK_URL, "foldseek.tar.gz")
    with tarfile.open("foldseek.tar.gz", "r:gz") as tar:
        tar.extractall()
    print("Foldseek installed successfully.")

# Setup Foldseek executable path
FOLDSEEK = pathlib.Path("foldseek/bin/foldseek").resolve()
!chmod +x {FOLDSEEK}
!{FOLDSEEK} version



### Fetch Reference PDBs: IsPETase (6EQE) and LCC (4EB0)
We fetch experimentally validated crystallographic structures:
- **IsPETase:** PDB ID `6EQE` (Chain A, 1.58 Å resolution)
- **LCC (Leaf-branch Compost Cutinase):** PDB ID `4EB0`


In [ ]:
REFS_DIR = pathlib.Path("references")
REFS_DIR.mkdir(exist_ok=True)

refs = {
    "6EQE": "https://files.rcsb.org/download/6EQE.pdb",
    "4EB0": "https://files.rcsb.org/download/4EB0.pdb"
}

for pdb_id, url in refs.items():
    dest = REFS_DIR / f"{pdb_id}.pdb"
    if not dest.exists():
        print(f"Fetching {pdb_id} from RCSB PDB...")
        urllib.request.urlretrieve(url, dest)
        print(f"Saved {dest}")

print("Reference PDBs ready in:", list(REFS_DIR.glob("*.pdb")))



## 3. Load the 71 Novel De Novo Designs


In [ ]:
from Bio import SeqIO
import pandas as pd

DESIGNS_PATH = pathlib.Path("designs_71_novel.faa")

# If running directly in Colab and file is in GitHub or local folder:
if not DESIGNS_PATH.exists():
    # Fallback: check workspace or download
    alt_paths = [
        pathlib.Path("experiments/experiment1_initial_tests/designs_71_novel.faa"),
        pathlib.Path("/content/drive/MyDrive/evo2_petase/designs_71_novel.faa")
    ]
    for p in alt_paths:
        if p.exists():
            shutil.copy(p, DESIGNS_PATH)
            break

records = list(SeqIO.parse(DESIGNS_PATH, "fasta"))
print(f"Loaded {len(records)} novel designs from {DESIGNS_PATH}")

df_designs = []
for r in records:
    # Header format: >petase_4_seq_20 n=4 max_pid=73.6 len=260
    parts = r.description.split()
    meta = {p.split("=")[0]: p.split("=")[1] for p in parts[1:] if "=" in p}
    df_designs.append({
        "id": r.id,
        "sequence": str(r.seq),
        "length": len(r.seq),
        "n_examples": int(meta.get("n", 0)),
        "max_pid": float(meta.get("max_pid", 0.0))
    })

df_designs = pd.DataFrame(df_designs)
print(df_designs.head())
print("\nBreakdown by in-context n_examples:")
print(df_designs["n_examples"].value_counts().sort_index())



## 4. Run ESMFold Prediction & Extract pLDDT
We use `facebook/esmfold_v1` via HuggingFace `transformers` on GPU (or ESM Atlas API as fallback).
Each design's 3D coordinates are saved to `esmfold_pdbs/`.
The mean pLDDT is calculated from the B-factor column.


In [ ]:
import torch
from transformers import AutoTokenizer, EsmForProteinFolding
import time

PDB_DIR = pathlib.Path("esmfold_pdbs")
PDB_DIR.mkdir(exist_ok=True)

print("Loading ESMFold v1 model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
model = model.to(device)
model.eval()
print(f"ESMFold loaded on {device}.")

# Function to fold a single sequence
def fold_sequence(seq, qid):
    pdb_path = PDB_DIR / f"{qid}.pdb"
    if pdb_path.exists():
        # Already folded
        return pdb_path.read_text()
    
    with torch.no_grad():
        inputs = tokenizer([seq], return_tensors="pt", add_special_tokens=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        pdb_str = model.output_to_pdb(outputs)[0]
        pdb_path.write_text(pdb_str)
        return pdb_str

# Run folding on all 71 designs
print(f"Folding {len(df_designs)} designs with ESMFold...")
t0 = time.time()
plddts = []

for idx, row in df_designs.iterrows():
    qid = row["id"]
    seq = row["sequence"]
    pdb_content = fold_sequence(seq, qid)
    
    # Extract pLDDT from PDB B-factors (columns 61-66 in ATOM lines)
    bfactors = []
    for line in pdb_content.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            try:
                bfactors.append(float(line[60:66].strip()))
            except:
                pass
    mean_plddt = sum(bfactors) / max(len(bfactors), 1)
    plddts.append(round(mean_plddt, 2))
    
    if (idx + 1) % 10 == 0 or (idx + 1) == len(df_designs):
        print(f"  [{idx+1:>2}/{len(df_designs)}] {qid:<20} len={len(seq)} mean_pLDDT={mean_plddt:.1f} ({time.time() - t0:.1f}s)")

df_designs["mean_plddt"] = plddts
df_designs["passed_plddt80"] = df_designs["mean_plddt"] >= 80.0
print(f"\nTotal folding time: {time.time() - t0:.1f}s")
print(f"Passed pLDDT >= 80.0: {df_designs['passed_plddt80'].sum()} / {len(df_designs)} ({100*df_designs['passed_plddt80'].mean():.1f}%)")



### Filter High-Confidence Designs & Summary


In [ ]:
import matplotlib.pyplot as plt

df_plddt80 = df_designs[df_designs["passed_plddt80"]].copy()
df_designs.to_csv("esmfold_plddt_all_71.tsv", sep="\t", index=False)
df_plddt80.to_csv("esmfold_plddt_passed_80.tsv", sep="\t", index=False)

# Export passed FASTA
with open("designs_passed_plddt80.faa", "w") as f:
    for _, row in df_plddt80.iterrows():
        f.write(f">{row['id']} plddt={row['mean_plddt']} n={row['n_examples']}\n{row['sequence']}\n")

print(f"Saved {len(df_plddt80)} high-confidence designs (pLDDT >= 80) to designs_passed_plddt80.faa")

# Plot pLDDT distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.hist(df_designs["mean_plddt"], bins=15, color="#2a78d6", edgecolor="black", alpha=0.8)
ax1.axvline(80.0, color="crimson", linestyle="--", label="Threshold = 80.0")
ax1.set_xlabel("Mean pLDDT")
ax1.set_ylabel("Count")
ax1.set_title("ESMFold Confidence Distribution (71 Designs)")
ax1.legend()

# Pass rate by in-context n_examples
pass_by_n = df_designs.groupby("n_examples")["passed_plddt80"].mean() * 100
ax2.bar([str(n) for n in pass_by_n.index], pass_by_n.values, color="#34a853", edgecolor="black", alpha=0.8)
ax2.set_xlabel("In-Context Example Count (n)")
ax2.set_ylabel("Pass Rate (% with pLDDT >= 80)")
ax2.set_title("Structural Quality Gate Pass Rate vs Context Size")
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig("esmfold_plddt_summary.png", dpi=200)
plt.show()



## 5. Foldseek Alignment vs IsPETase (6EQE) & LCC (4EB0)
We align all predicted PDB structures against the reference crystallographic models using Foldseek TM-score mode.


In [ ]:
# Run Foldseek easy-search
ALIGN_OUT = pathlib.Path("foldseek_vs_references.m8")
TMP_DIR = pathlib.Path("tmp_foldseek")
TMP_DIR.mkdir(exist_ok=True)

FIELDS = "query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,alntmscore,rmsd,lddt"

cmd = [
    str(FOLDSEEK), "easy-search",
    str(PDB_DIR), str(REFS_DIR),
    str(ALIGN_OUT), str(TMP_DIR),
    "--format-output", FIELDS,
    "-e", "1.0",
    "--alignment-type", "1"   # TM-score alignment mode
]

print("Running Foldseek structural search...")
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("Foldseek stderr:", res.stderr)
else:
    print(f"Foldseek completed successfully! Alignment rows: {sum(1 for _ in open(ALIGN_OUT))}")

# Parse Foldseek results
df_fs = pd.read_csv(ALIGN_OUT, sep="\t", names=FIELDS.split(","))
# Clean query name (strip .pdb suffix)
df_fs["query"] = df_fs["query"].str.replace(".pdb", "", regex=False)
df_fs["target"] = df_fs["target"].str.replace(".pdb", "", regex=False)

print("\nTop alignments vs 6EQE (IsPETase):")
print(df_fs[df_fs["target"] == "6EQE"].sort_values(by="alntmscore", ascending=False)[["query", "target", "fident", "alntmscore", "rmsd", "evalue"]].head(10))

print("\nTop alignments vs 4EB0 (LCC):")
print(df_fs[df_fs["target"] == "4EB0"].sort_values(by="alntmscore", ascending=False)[["query", "target", "fident", "alntmscore", "rmsd", "evalue"]].head(10))



## 6. Check Active Site in 3D: Catalytic Triad Geometry (Ser - His - Asp)
In PETases and cutinases (IsPETase `6EQE`, LCC `4EB0`):
- **Serine (Nucleophile):** S160 in 6EQE, located in the conserved motif `[G/W/A]-x-S-x-G`
- **Histidine (General Base):** H237 in 6EQE
- **Aspartate (Acid/Relay):** D206 in 6EQE

**Geometric Acceptance Constraints:**
1. $\text{dist}(\text{Ser } \text{O}\gamma, \text{His } \text{N}\epsilon2) \le 3.5\text{ \AA}$ (Activation of nucleophile)
2. $\text{dist}(\text{His } \text{N}\delta1, \text{Asp } \text{O}\delta) \le 3.2\text{ \AA}$ (Proton relay hydrogen bond)
3. $\text{dist}(\text{Ser } \text{O}\gamma, \text{Asp } \text{C}\gamma) \le 7.0\text{ \AA}$ (Triad spatial clustering)


In [ ]:
from Bio.PDB import PDBParser
import numpy as np
import re

parser = PDBParser(QUIET=True)

def find_catalytic_triad(structure, sequence):
    # 1. Locate nucleophile Serine via G-x-S-x-G motif
    # Search for regex: [GWA][A-Z]S[A-Z]G
    motif_match = re.search(r'[GWA].[SMG].G', sequence)
    ser_idx = None
    if motif_match:
        ser_idx = motif_match.start() + 2  # 0-indexed position of Ser
    else:
        # Fallback to standard G-x-S
        m2 = re.search(r'G.S', sequence)
        if m2:
            ser_idx = m2.start() + 2

    # Residues list
    model = structure[0]
    chain = list(model.get_chains())[0]
    residues = [r for r in chain if r.id[0] == ' ']
    
    if ser_idx is None or ser_idx >= len(residues):
        return None
        
    ser_res = residues[ser_idx]
    if ser_res.get_resname() != "SER" or "OG" not in ser_res:
        return None
        
    ser_og = ser_res["OG"].get_coord()
    
    # 2. Search for candidate His (downstream, usually within 10 Å of Ser OG)
    best_pair = None
    min_ser_his_dist = 999.0
    
    for r in residues:
        if r.get_resname() == "HIS" and "NE2" in r and "ND1" in r:
            his_ne2 = r["NE2"].get_coord()
            dist = np.linalg.norm(ser_og - his_ne2)
            if dist < min_ser_his_dist:
                min_ser_his_dist = dist
                best_his = r

    if min_ser_his_dist > 8.0:  # No His anywhere near the active site
        return None
        
    his_res = best_his
    his_ne2 = his_res["NE2"].get_coord()
    his_nd1 = his_res["ND1"].get_coord()
    
    # 3. Search for candidate Asp near His ND1
    best_asp = None
    min_his_asp_dist = 999.0
    for r in residues:
        if r.get_resname() == "ASP" and ("OD1" in r or "OD2" in r):
            asp_coord = r["OD1"].get_coord() if "OD1" in r else r["OD2"].get_coord()
            dist1 = np.linalg.norm(his_nd1 - r["OD1"].get_coord()) if "OD1" in r else 999
            dist2 = np.linalg.norm(his_nd1 - r["OD2"].get_coord()) if "OD2" in r else 999
            d_min = min(dist1, dist2)
            if d_min < min_his_asp_dist:
                min_his_asp_dist = d_min
                best_asp = r
                
    if best_asp is None:
        return None
        
    asp_res = best_asp
    asp_od1 = asp_res["OD1"].get_coord() if "OD1" in asp_res else asp_res["OD2"].get_coord()
    asp_cg = asp_res["CG"].get_coord() if "CG" in asp_res else asp_od1
    
    ser_asp_dist = np.linalg.norm(ser_og - asp_cg)
    
    return {
        "ser_id": f"{ser_res.get_resname()}{ser_res.id[1]}",
        "his_id": f"{his_res.get_resname()}{his_res.id[1]}",
        "asp_id": f"{asp_res.get_resname()}{asp_res.id[1]}",
        "d_ser_his": round(float(min_ser_his_dist), 2),
        "d_his_asp": round(float(min_his_asp_dist), 2),
        "d_ser_asp": round(float(ser_asp_dist), 2),
        "triad_intact": (min_ser_his_dist <= 3.5) and (min_his_asp_dist <= 3.2)
    }

# Run triad check across all designs with pLDDT >= 80
triad_results = []
for idx, row in df_plddt80.iterrows():
    qid = row["id"]
    pdb_path = PDB_DIR / f"{qid}.pdb"
    structure = parser.get_structure(qid, str(pdb_path))
    triad = find_catalytic_triad(structure, row["sequence"])
    
    res = {
        "id": qid,
        "n_examples": row["n_examples"],
        "mean_plddt": row["mean_plddt"],
        "max_pid": row["max_pid"]
    }
    if triad:
        res.update(triad)
    else:
        res.update({
            "ser_id": "None", "his_id": "None", "asp_id": "None",
            "d_ser_his": 99.0, "d_his_asp": 99.0, "d_ser_asp": 99.0,
            "triad_intact": False
        })
    triad_results.append(res)

df_triads = pd.DataFrame(triad_results)
df_triads.to_csv("catalytic_triad_geometry_report.tsv", sep="\t", index=False)

intact_count = df_triads["triad_intact"].sum()
print(f"Catalytic Triad Check Complete:")
print(f"  Intact 3D Triad (Ser-His <= 3.5A, His-Asp <= 3.2A): {intact_count} / {len(df_triads)}")

print("\nTop Candidates (High pLDDT + Intact Catalytic Triad):")
top_candidates = df_triads[df_triads["triad_intact"]].sort_values(by="mean_plddt", ascending=False)
print(top_candidates[["id", "n_examples", "mean_plddt", "max_pid", "ser_id", "his_id", "asp_id", "d_ser_his", "d_his_asp"]].to_string(index=False))



## 7. Statistical Scaling: Re-run with $N\_PROMPTS = 25$
To obtain tight error bars matching the published benchmark:
- Set `N_PROMPTS = 25` (with `SAMPLES_PER_PROMPT = 4` $\rightarrow$ **100 samples per cell**, 600 total across $n \in \{2, 4, 8, 16, 32, 64\}$).
- **Memory Boundary:**
  - $n=64$ requires $\approx 53\text{ kb}$ DNA context, which fits in 24 GB VRAM (L4/A100) using FP8 KV caching.
  - $n=128$ requires $> 106\text{ kb}$ DNA context, exceeding 24 GB VRAM (`OOM`).
  - Therefore, we keep the grid up to $n=64$ and omit $n=128$.


In [ ]:
# Configuration snippet for exp1_initial_discovery.ipynb
N_PROMPTS          = 25      # scaled from 10 to 25 for tighter SEM error bars
SAMPLES_PER_PROMPT = 4       # -> 100 samples per cell
MAX_ID             = 85.0

print(f"Configured N_PROMPTS = {N_PROMPTS}. Total generation volume: {6 * N_PROMPTS * SAMPLES_PER_PROMPT} sequences across 6 grid cells.")



## 8. Activity-Biased In-Context Conditioning (Step 6)
Instead of sampling uniformly from the random pool, we condition Evo 2 strictly on **thermotolerant** ($T_{opt} \ge 50-70^\circ\text{C}$) and **high-degradation** PETase exemplars from PANDA:
- **LCC** ($T_{opt} = 65^\circ\text{C}$, degradation $> 27,000\ \mu\text{M}$)
- **TfCut2 / TfCut1** ($T_{opt} = 60^\circ\text{C}$, degradation $> 12,000 - 16,000\ \mu\text{M}$)
- **BhrPETase** ($T_{opt} = 70^\circ\text{C}$)
- **611, 701, 716, 504_CtPL, bbPET0069**


In [ ]:
# Function to sample examples strictly from the curated elite pool
import random

def select_activity_biased_examples(activity_pool, n, seed):
    random.seed(seed)
    # Sort or weight by activity/temperature if desired, or sample from elite set
    return random.sample(activity_pool, min(n, len(activity_pool)))

print("Activity-biased selection function defined.")

